In [8]:
%pip install -U transformers datasets accelerate evaluate scikit-learn

In [9]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
Transformers: 5.23.0
Datasets: 5.2.0
Device: cuda
GPU: Tesla T4


In [10]:
from datasets import load_dataset

dataset = load_dataset("nyu-mll/glue", "sst2")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [11]:
for sample in dataset["train"].select(range(5)):
    print("Sentence:", sample["sentence"])
    print("Label:", sample["label"])
    print("-" * 50)

Sentence: hide new secretions from the parental units 
Label: 0
--------------------------------------------------
Sentence: contains no wit , only labored gags 
Label: 0
--------------------------------------------------
Sentence: that loves its characters and communicates something rather beautiful about human nature 
Label: 1
--------------------------------------------------
Sentence: remains utterly satisfied to remain the same throughout 
Label: 0
--------------------------------------------------
Sentence: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
Label: 0
--------------------------------------------------


In [12]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

text = "I really enjoyed learning about Transformers."

tokens = tokenizer(
    text,
    truncation=True,
    max_length=128
)

print(tokens)
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"]))

{'input_ids': [101, 1045, 2428, 5632, 4083, 2055, 19081, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
['[CLS]', 'i', 'really', 'enjoyed', 'learning', 'about', 'transformers', '.', '[SEP]']


In [13]:
def tokenize_function(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=128
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["sentence", "idx"]
)

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})


In [14]:
from transformers import AutoModelForSequenceClassification

AutoModelForSequenceClassification

transformers.models.auto.modeling_auto.AutoModelForSequenceClassification

In [15]:
from transformers import AutoModelForSequenceClassification

id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print(model)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [16]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [17]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    f1 = f1_score(
        labels,
        predictions,
        average="binary"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert-sst2-results",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=2,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    logging_steps=50,
    save_total_limit=2,

    report_to="none",
    seed=42
)

In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [20]:
train_result = trainer.train()

print(train_result)

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.181763,0.314655,0.909404,0.910734
2,0.098167,0.358212,0.905963,0.908889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8420, training_loss=0.1769157838934674, metrics={'train_runtime': 493.152, 'train_samples_per_second': 273.137, 'train_steps_per_second': 17.074, 'total_flos': 1226900886258636.0, 'train_loss': 0.1769157838934674, 'epoch': 2.0})


For each epoch:
    For each batch:
        1. Load tokenized input
        2. Forward pass through DistilBERT
        3. Calculate classification loss
        4. Backpropagate gradients
        5. Update model parameters

    Evaluate on validation data
    Save checkpoint



In [21]:
evaluation_results = trainer.evaluate()

print("Validation results:")

for metric, value in evaluation_results.items():
    print(f"{metric}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.098167,0.314655,2,0.909404,0.910734


Validation results:
eval_loss: 0.3147
eval_accuracy: 0.9094
eval_f1: 0.9107


In [22]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

predictions = trainer.predict(
    tokenized_dataset["validation"]
)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=["Negative", "Positive"]
    )
)

print(
    confusion_matrix(
        true_labels,
        predicted_labels
    )
)

              precision    recall  f1-score   support

    Negative       0.90      0.91      0.91       428
    Positive       0.91      0.91      0.91       444

    accuracy                           0.91       872
   macro avg       0.91      0.91      0.91       872
weighted avg       0.91      0.91      0.91       872

[[390  38]
 [ 41 403]]


In [23]:
import torch
import torch.nn.functional as F


def predict_sentiment(text, model, tokenizer):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # Place inputs on the model's device
    model_device = next(model.parameters()).device
    inputs = {
        key: value.to(model_device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)

    probabilities = F.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_class = probabilities.argmax(
        dim=-1
    ).item()

    confidence = probabilities[0, predicted_class].item()

    label = model.config.id2label[predicted_class]

    return {
        "text": text,
        "sentiment": label,
        "confidence": round(confidence, 4)
    }

In [24]:
test_sentences = [
    "This movie was absolutely fantastic!",
    "The story was boring and disappointing.",
    "I enjoyed every moment of this experience.",
    "The acting was terrible."
]

for sentence in test_sentences:
    result = predict_sentiment(
        sentence,
        trainer.model,
        tokenizer
    )

    print(result)

{'text': 'This movie was absolutely fantastic!', 'sentiment': 'POSITIVE', 'confidence': 0.9989}
{'text': 'The story was boring and disappointing.', 'sentiment': 'NEGATIVE', 'confidence': 0.9978}
{'text': 'I enjoyed every moment of this experience.', 'sentiment': 'POSITIVE', 'confidence': 0.9979}
{'text': 'The acting was terrible.', 'sentiment': 'NEGATIVE', 'confidence': 0.9977}


In [25]:
SAVE_DIR = "./distilbert-sst2-final"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model and tokenizer saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully.


In [26]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

saved_tokenizer = AutoTokenizer.from_pretrained(
    SAVE_DIR
)

saved_model = AutoModelForSequenceClassification.from_pretrained(
    SAVE_DIR
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]